The **Intelligence**

In [1]:
from datetime import datetime, timedelta

# --- [CP-01] Parametric Observation Window ---

def get_observation_window():
    """FUNCTION 1: Get the 'Memory' parameter (weeks)"""
    # The Crux: We start with a 4-week baseline.
    # This is where the AI decides how far back to look.
    active_window_weeks = 4
    return active_window_weeks

def calculate_date_boundary(weeks):
    """FUNCTION 2: Calculate the 'Cut-off' date"""
    # The Crux: Today is March 1, 2026.
    # This function finds the date 28 days ago.
    today = datetime.now()
    start_date = today - timedelta(weeks=weeks)
    return start_date

# --- TEST THE LOGIC ---
window = get_observation_window()
boundary = calculate_date_boundary(window)

print(f"--- Your Partner in Kitchen: Memory Status ---")
print(f"Memory Window: {window} weeks")
print(f"Currently ignoring anything before: {boundary.strftime('%B %d, %Y')}")

--- Your Partner in Kitchen: Memory Status ---
Memory Window: 4 weeks
Currently ignoring anything before: February 01, 2026


In [2]:
# --- [CP-01] User Intelligence Profile Logic ---

class UserIntelligenceProfile:
    def __init__(self, user_id, active_window_size=4):
        self.user_id = user_id
        # AC: active_window_size (Integer) created
        self.active_window_size = active_window_size

    def get_window_start_date(self):
        """AC: System calculates $W_{obs}$ boundary for Sunday drafts"""
        today = datetime.now()
        # Logic: Current Date - (active_window_size * 7 days)
        w_obs_boundary = today - timedelta(weeks=self.active_window_size)
        return w_obs_boundary

# --- IMPLEMENTATION ---
# Imagine this is fetching from your backend config
profile = UserIntelligenceProfile(user_id="User_01", active_window_size=4)
w_obs = profile.get_window_start_date()

print(f"✅ AC 1: User_Intelligence_Profile initialized with size {profile.active_window_size}")
print(f"✅ AC 2: Sunday Drafts will ignore data older than: {w_obs.strftime('%Y-%m-%d')}")

✅ AC 1: User_Intelligence_Profile initialized with size 4
✅ AC 2: Sunday Drafts will ignore data older than: 2026-02-01


In [3]:
# --- BASELINE: [CP-01] Parametric Window ---

def run_sprint_1_check():
    window = get_observation_window()
    boundary = calculate_date_boundary(window)

    print(f"✅ User Story [CP-01] Closed.")
    print(f"Decision: The Partner will look back to {boundary.strftime('%Y-%m-%d')}.")
    print(f"Status: Logic is ready for Function 3 (Data Fetching).")

run_sprint_1_check()

✅ User Story [CP-01] Closed.
Decision: The Partner will look back to 2026-02-01.
Status: Logic is ready for Function 3 (Data Fetching).


logic for function 3 - rolling history

In [4]:
from datetime import datetime, timedelta
import json

# --- FN 2: CALCULATE BOUNDARY (The logic we baselined earlier) ---
def calculate_boundary(active_window_size=4):
    today = datetime.now()
    # Logic: Current Date - (W_obs * 7 days)
    return today - timedelta(weeks=active_window_size)

In [5]:
def fetch_rolling_history(user_id, start_date):
    vault_data = [
        {"meal_id": "M-101", "date": datetime(2026, 2, 25), "complexity_score": 8, "regional_tags": ["North Indian"]},
        {"meal_id": "M-102", "date": datetime(2026, 2, 20), "complexity_score": 4, "regional_tags": ["Continental"]},
        {"meal_id": "M-099", "date": datetime(2026, 1, 1), "complexity_score": 9, "regional_tags": ["Italian"]}
    ]

    # Filter and convert datetime to string so JSON can handle it
    active_history = []
    for item in vault_data:
        if item["date"] >= start_date:
            # We create a copy and convert the date to a string
            clean_item = item.copy()
            clean_item["date"] = item["date"].isoformat() # <--- THE FIX
            active_history.append(clean_item)

    return json.dumps(active_history)

# execution flow

In [6]:
# --- THE EXECUTION FLOW ---
# 1. Get the boundary first
w_obs_boundary = calculate_boundary(active_window_size=4)

# 2. Now pass it to the fetcher
rolling_json = fetch_rolling_history("Lead_Dev_01", w_obs_boundary)

print(f"✅ Boundary set to: {w_obs_boundary.strftime('%Y-%m-%d')}")
print(f"📦 Successfully fetched {len(json.loads(rolling_json))} records for analysis.")

✅ Boundary set to: 2026-02-01
📦 Successfully fetched 2 records for analysis.


In [7]:
# --- [CP-03] Function 4: Taste DNA Analysis ---

def analyze_taste_dna(json_data):
    """
    Function 3.1: Parses rolling history to find trends.
    Output: DNA Dictionary (Avg Complexity, Top Regions)
    """
    data = json.loads(json_data)

    if not data:
        return {"avg_complexity": 0, "top_regions": [], "status": "No data in window"}

    # 1. Calculate Average Complexity
    total_complexity = sum(item['complexity_score'] for item in data)
    avg_complexity = total_complexity / len(data)

    # 2. Extract and Flatten Regional Tags
    all_tags = []
    for item in data:
        all_tags.extend(item['regional_tags'])

    # 3. Find unique regions (Dominant Tags)
    unique_regions = list(set(all_tags))

    dna_profile = {
        "avg_complexity": round(avg_complexity, 2),
        "dominant_regions": unique_regions,
        "meal_count": len(data)
    }

    return dna_profile

# --- EXECUTION FLOW ---
# Pass the output from Function 3 into Function 4
user_dna = analyze_taste_dna(rolling_json)

print("--- 🧬 TASTE DNA PROFILE GENERATED ---")
print(f"Current Complexity Level: {user_dna['avg_complexity']}/10")
print(f"Active Regional Interests: {', '.join(user_dna['dominant_regions'])}")

--- 🧬 TASTE DNA PROFILE GENERATED ---
Current Complexity Level: 6.0/10
Active Regional Interests: Continental, North Indian


In [8]:
# --- [CP-03] Function 4: Behavior Drift Detector ---

def identify_behavior_drift(json_data):
    """
    Function 4: Detects significant shifts in cooking habits.
    If complexity changes by > 30%, it triggers a 'Drift'.
    """
    data = json.loads(json_data)
    if len(data) < 2: return False

    # 1. Split data into Current Week (Latest) and History (Previous)
    # (Assuming data is sorted by date)
    current_week = data[0:1] # Most recent meal
    previous_weeks = data[1:] # Older meals in window

    # 2. Compare Average Complexity
    curr_comp = sum(m['complexity_score'] for m in current_week) / len(current_week)
    prev_comp = sum(m['complexity_score'] for m in previous_weeks) / len(previous_weeks)

    # 3. Logic: If complexity shift is > 30%, habits have drifted
    drift_threshold = 0.30
    percent_change = abs(curr_comp - prev_comp) / prev_comp

    has_drifted = percent_change > drift_threshold

    if has_drifted:
        print("⚠️ DRIFT DETECTED: User habits are shifting significantly.")
    else:
        print("✅ STABLE: Habits are consistent with history.")

    return has_drifted

# --- EXECUTION ---
drift_signal = identify_behavior_drift(rolling_json)

⚠️ DRIFT DETECTED: User habits are shifting significantly.


In [9]:
# --- [CP-00] Function 5: Intelligence Profile Writer ---

def update_intelligence_profile(user_id, new_window_size, recalc_trigger):
    """
    Function 5: The 'Writer'. Saves parameters to the database.
    Input: user_id, new_window_size (the updated W_obs), recalc_trigger (timestamp)
    """

    # In production, this would be:
    # db.execute("UPDATE Profiles SET window = ?, last_recalc = ? WHERE id = ?", ...)

    intelligence_payload = {
        "user_id": user_id,
        "active_window_size": f"{new_window_size} weeks",
        "last_recalc_timestamp": recalc_trigger.strftime('%Y-%m-%d %H:%M:%S'),
        "status": "COMPLETED"
    }

    # Simulation of a successful DB write
    print(f"💾 DATABASE UPDATE SUCCESSFUL for {user_id}")
    print(f"📝 New Parameters: Window={new_window_size}, TriggeredAt={intelligence_payload['last_recalc_timestamp']}")

    return None # Output is void as per spec

# --- FINAL EXECUTION FLOW ---

# If drift was detected in Fn 4, we suggest a shorter window (e.g., 2 weeks)
suggested_window = 2 if drift_signal else 4

update_intelligence_profile(
    user_id="Vramanbalahm",
    new_window_size=suggested_window,
    recalc_trigger=datetime.now()
)

💾 DATABASE UPDATE SUCCESSFUL for Vramanbalahm
📝 New Parameters: Window=2, TriggeredAt=2026-03-01 12:05:00


In [14]:
# --- [CP-02] Function 6: Confidence Engine ---

def calculate_confidence_score(suggested_plan, final_export):
    """
    Compares Suggested vs Final to update current_confidence_score (0.0 - 1.0).
    Triggers 'Discovery Mode' vs 'Smart Draft'.
    """
    # 1. Calculate Match Rate
    suggested_set = set(suggested_plan)
    final_set = set(final_export)

    matches = len(suggested_set.intersection(final_set))
    total_suggested = len(suggested_set)

    if total_suggested == 0: return 0.0

    # CS = Match Ratio
    cs = matches / total_suggested

    # 2. Determine UI Mode
    if cs > 0.7:
        ui_mode = "Smart Draft (High Trust)"
    elif cs < 0.4:
        ui_mode = "Discovery Mode (Learning)"
    else:
        ui_mode = "Standard"

    # 3. Detect Sudden Drops (Behavior Drift / Momentum Divergence)
    # Using your saved logic: If CS drops suddenly, log Recalibration
    recalibration_event = cs < 0.3  # Threshold for a sudden peak/drop

    return {
        "current_confidence_score": round(cs, 2),
        "ui_mode": ui_mode,
        "recalibration_required": recalibration_event
    }

# --- TEST THE FEEDBACK LOOP ---
suggestion = ["Paneer Tikka", "Pasta", "Salad", "Dal Tadka"]
final_user_choice = ["Paneer Tikka", "Ramen", "Steak", "Dal Tadka"] # 50% change

results = calculate_confidence_score(suggestion, final_user_choice)

print(f"📊 Confidence Score: {results['current_confidence_score']}")
print(f"🖥️ UI State: {results['ui_mode']}")
if results['recalibration_required']:
    print("🚨 LOGGED: Recalibration Event - Significant Behavior Shift detected.")

📊 Confidence Score: 0.5
🖥️ UI State: Standard


In [15]:
# --- [CP-03] Schema Update: Intelligence-Ready Vault ---

def batch_update_vault_metadata(vault_list):
    """
    Simulates a batch update of 1,000+ recipes.
    Assigns complexity and regional bias while preserving image URLs.
    """
    updated_vault = []

    for recipe in vault_list:
        # 1. Preserving existing Image Assets
        hero = recipe.get('hero_image')
        thumb = recipe.get('carousel_thumb')

        # 2. Logic-Based Tagging (Example: "Dal" -> North, 20 mins -> Low Complexity)
        # In production, this would be an AI-tagging pass.
        complexity = 3 if "Quick" in recipe['title'] else 7
        region = "North Indian" if "Dal" in recipe['title'] else "Continental"

        # 3. Enriching the Record
        enriched_recipe = {
            **recipe,
            "complexity_score": complexity,
            "regional_bias": region,
            "hero_image": hero,
            "carousel_thumb": thumb
        }
        updated_vault.append(enriched_recipe)

    return updated_vault

# --- EXECUTION ---
# Mocking a snippet of the 1,000 recipes
raw_vault = [
    {"title": "Quick Dal Fry", "hero_image": "url_a", "carousel_thumb": "thumb_a"},
    {"title": "Slow Cooked Risotto", "hero_image": "url_b", "carousel_thumb": "thumb_b"}
]

intelligence_ready_vault = batch_update_vault_metadata(raw_vault)
print(f"✅ Batch Update Complete: {len(intelligence_ready_vault)} recipes enriched.")
print(f"Sample Entry: {intelligence_ready_vault[0]['title']} | Complexity: {intelligence_ready_vault[0]['complexity_score']}")

✅ Batch Update Complete: 2 recipes enriched.
Sample Entry: Quick Dal Fry | Complexity: 3


In [16]:
# --- [CP-03]Recipe Complexity ($C$) & Regional Tagging,  Granular Schema Update ---

def enrich_vault_with_states(recipe_list):
    """
    Updates the vault to use State-level granularity.
    Ensures 'regional_bias' is a list of specific origins.
    """
    for recipe in recipe_list:
        title = recipe['title'].lower()

        # State-level mapping logic
        if "dosa" in title or "idli" in title:
            recipe['regional_bias'] = ["Tamil Nadu", "Karnataka"]
        elif "appam" in title or "avial" in title:
            recipe['regional_bias'] = ["Kerala"]
        elif "dal baati" in title:
            recipe['regional_bias'] = ["Rajasthan"]
        else:
            recipe['regional_bias'] = ["General/Continental"]

        # Complexity remains (1-10)
        recipe['complexity_score'] = recipe.get('complexity_score', 5)

    return recipe_list

In [17]:
# --- [CP-03] Migration: State-Level Intelligence Update ---

def migrate_vault_to_state_intelligence(vault_data):
    """
    Batch updates the Recipe_Content_Vault.
    Preserves: hero_image, carousel_thumb.
    Adds: complexity_score (1-10), regional_bias (State Array).
    """
    # 1. State Mapping Dictionary (The "State Anchors")
    state_map = {
        "dosa": ["Tamil Nadu", "Karnataka"],
        "idli": ["Tamil Nadu"],
        "appam": ["Kerala"],
        "avial": ["Kerala"],
        "poha": ["Maharashtra", "MP"],
        "misal": ["Maharashtra"],
        "dal baati": ["Rajasthan"],
        "pasta": ["Italian/Continental"]
    }

    enriched_vault = []

    for recipe in vault_data:
        title_lower = recipe['title'].lower()

        # 2. Logic: Assign State Tags based on Title Keywords
        tags = ["General"] # Default
        for key, states in state_map.items():
            if key in title_lower:
                tags = states
                break

        # 3. Logic: Assign Complexity Score (1-10)
        # We assume 'Quick' or 'Instant' are 1-3, others are standard 5-6
        complexity = 3 if any(x in title_lower for x in ["quick", "instant", "poha"]) else 6

        # 4. Construct the Enriched Record (Preserving Images)
        enriched_record = {
            "title": recipe['title'],
            "hero_image": recipe['hero_image'], # PRESERVED
            "carousel_thumb": recipe['carousel_thumb'], # PRESERVED
            "complexity_score": complexity,
            "regional_bias": tags, # NEW ARRAY
            "prep_steps": recipe.get('prep_steps', "Steps in Vault...")
        }
        enriched_vault.append(enriched_record)

    return enriched_vault

# --- TEST THE MIGRATION ---
mock_vault = [
    {"title": "Instant Rava Dosa", "hero_image": "storage/hero_1.jpg", "carousel_thumb": "storage/thumb_1.jpg"},
    {"title": "Kerala Appam", "hero_image": "storage/hero_2.jpg", "carousel_thumb": "storage/thumb_2.jpg"}
]

final_vault = migrate_vault_to_state_intelligence(mock_vault)
print(f"✅ Migration Complete. Sample: {final_vault[0]['title']} -> {final_vault[0]['regional_bias']}")

✅ Migration Complete. Sample: Instant Rava Dosa -> ['Tamil Nadu', 'Karnataka']


In [19]:
# --- [CP-05] Refined Function 7: Dynamic Stress Pivot ---

def monitor_market_pivot(current_market_data, current_meal):
    """
    Dynamically monitors if any tracked asset hits its specific
    Wave 5 target while RSI diverges.
    """
    # 1. Dynamic Check (Works for 1277 or any other Target)
    price = current_market_data['price']
    target = current_market_data['target_price']
    rsi_now = current_market_data['rsi']
    rsi_prev = current_market_data['rsi_prev']

    # The Logic: Price hits/exceeds target BUT momentum (RSI) is failing
    is_momentum_divergence = (price >= target) and (rsi_now < rsi_prev)

    if is_momentum_divergence:
        # 2. Check Complexity (C >= 7)
        if current_meal['complexity_score'] >= 7:
            # 3. Trigger the State-Specific One-Pot Swap
            return {
                "signal": "PIVOT",
                "reason": f"Target {target} hit with RSI Divergence",
                "swap_to": "Complexity <= 3",
                "region": current_meal['regional_bias']
            }

    return {"signal": "STABLE"}

# --- EXAMPLE DATA ---

# 1. Define the 'complex_meal' (The high-effort recipe currently planned)
complex_meal = {
    "title": "Full Kerala Sadya",
    "complexity_score": 9,
    "regional_bias": ["Kerala"]
}

# 2. Define the 'market_feed' (The dynamic signal for any item)
market_feed = {
    "asset": "Item_A",
    "price": 1278,
    "target_price": 1277,
    "rsi": 62,
    "rsi_prev": 68
}

# 3. Now the function call will work
status = monitor_market_pivot(market_feed, complex_meal)

if status["signal"] == "PIVOT":
    print(f"🚀 PIVOT ACTIVE: Suggesting C <= 3 {status['region']} alternatives.")
else:
    print("✅ STATUS: System Stable.")


🚀 PIVOT ACTIVE: Suggesting C <= 3 ['Kerala'] alternatives.
